# Laboratorio 6 - Transfer Learning y Fine-Tuning
CC3092 Deep Learning y Sistemas Inteligentes

Modelos: CNN desde cero (32x32), VGG-16 como feature extractor y VGG-16 con fine-tuning.
Para VGG usamos 112x112 en vez de 224x224 por tiempo y memoria (GPU de 8 GB).

## 0. Setup

In [ ]:
import copy, json, os, platform, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
from torchvision.models import vgg16, VGG16_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from thop import profile

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
DEV = torch.device("cuda")
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
VGG_RES = 112
os.makedirs("results/figs", exist_ok=True)

HW = {"gpu": torch.cuda.get_device_name(0),
      "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
      "cpu": platform.processor() or platform.machine(), "torch": torch.__version__}
print(HW)

## 1-2. Dataset, exploración y preparación

In [ ]:
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
if os.path.exists("data/hf/train.parquet"):
    # copia de CIFAR-10 de huggingface (la descarga de torchvision estaba muy lenta)
    import io
    from PIL import Image
    def load_parquet(path):
        df = pd.read_parquet(path)
        X = np.stack([np.array(Image.open(io.BytesIO(d["bytes"])).convert("RGB")) for d in df["img"]])
        return X, df["label"].to_numpy()
    Xtr_all, ytr_all = load_parquet("data/hf/train.parquet")
    Xte, yte = load_parquet("data/hf/test.parquet")
else:
    train_raw = datasets.CIFAR10(root="data", train=True, download=True)
    test_raw = datasets.CIFAR10(root="data", train=False, download=True)
    Xtr_all, ytr_all = train_raw.data, np.array(train_raw.targets)
    Xte, yte = test_raw.data, np.array(test_raw.targets)
print("train:", Xtr_all.shape, "test:", Xte.shape, "clases:", len(CLASSES))
print("conteo por clase (train):", dict(zip(CLASSES, np.bincount(ytr_all))))
print("conteo por clase (test):", dict(zip(CLASSES, np.bincount(yte))))

Observaciones: 60,000 imágenes (50,000 train + 10,000 test), 10 clases perfectamente balanceadas (5,000/1,000 por clase). Imágenes RGB de 32×32 (3 canales).

In [ ]:
cifar_mean = Xtr_all.reshape(-1, 3).mean(0) / 255
cifar_std = Xtr_all.reshape(-1, 3).std(0) / 255
IMNET_MEAN, IMNET_STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
print(pd.DataFrame({"CIFAR mean": cifar_mean, "ImageNet mean": IMNET_MEAN,
                    "CIFAR std": cifar_std, "ImageNet std": IMNET_STD}, index=["R", "G", "B"]).round(4))

Las medias de CIFAR-10 (~0.49, 0.48, 0.45) son muy parecidas a las de ImageNet; las desviaciones estándar son algo mayores (~0.25 vs ~0.22). Para la CNN desde cero usamos las estadísticas propias de CIFAR; para VGG-16 debemos usar las de ImageNet, porque los filtros preentrenados esperan entradas con esa distribución.

In [ ]:
fig, axes = plt.subplots(10, 5, figsize=(6, 12))
rng = np.random.default_rng(SEED)
for c in range(10):
    idx = rng.choice(np.where(ytr_all == c)[0], 5, replace=False)
    for j, i in enumerate(idx):
        ax = axes[c, j]; ax.imshow(Xtr_all[i]); ax.axis("off")
        if j == 0: ax.set_title(CLASSES[c], fontsize=8, loc="left")
plt.tight_layout(); plt.savefig("results/figs/ejemplos.png", dpi=110); plt.show()

Pares difíciles esperados: *cat/dog* (misma pose, pelaje y fondo doméstico), *automobile/truck* (vehículos con ruedas y vistas similares), *deer/horse* (cuadrúpedos sobre pasto), *airplane/ship* y *airplane/bird* (fondos azules de cielo/agua dominan a 32×32). A tan baja resolución desaparecen los detalles finos que los separarían.

Pipelines de preprocesamiento
- CNN desde cero: 32×32 nativo -> `[0,1]` -> normalización con media/std de CIFAR. Train: random crop 32 con padding 4 (reflect) + flip horizontal.
- VGG-16: misma augmentation a 32×32 -> resize bilineal a 112×112 -> normalización ImageNet (equivalente a `VGG16_Weights.IMAGENET1K_V1.transforms()`, que hace resize 256 + center crop 224 + normalización ImageNet; reducimos a 112 por cómputo).

Costo del resize: 32×32 = 1,024 px; 224×224 = 50,176 px (×49); 112×112 = 12,544 px (×12.25). En una red convolucional el costo es proporcional al área espacial, así que pasar a 224 multiplica por ~49 los MACs de las capas conv y la memoria de activaciones; 112 lo reduce 4× respecto a 224.

Augmentation: random crop con padding (invarianza a pequeñas traslaciones) y flip horizontal (los objetos de CIFAR no cambian de clase al reflejarse). No usamos flips verticales ni rotaciones fuertes porque generan imágenes poco realistas (aviones/barcos al revés). La augmentation se aplica solo en train: su objetivo es regularizar aumentando la diversidad vista por el modelo; validación y test deben medir el desempeño sobre la distribución real y ser deterministas/comparables entre modelos.

In [ ]:
# split 45k/5k estratificado, test solo al final
idx_tr, idx_va = train_test_split(np.arange(len(ytr_all)), test_size=5000, stratify=ytr_all, random_state=SEED)
# 10% del train para la parte 4.1
idx_tr10, _ = train_test_split(idx_tr, train_size=0.10, stratify=ytr_all[idx_tr], random_state=SEED)
print(len(idx_tr), len(idx_va), len(idx_tr10), np.bincount(ytr_all[idx_tr10]))

def to_gpu(X, y):
    return (torch.from_numpy(X).permute(0, 3, 1, 2).contiguous().to(DEV),
            torch.from_numpy(y).long().to(DEV))

DATA = {"train": to_gpu(Xtr_all[idx_tr], ytr_all[idx_tr]),
        "val": to_gpu(Xtr_all[idx_va], ytr_all[idx_va]),
        "train10": to_gpu(Xtr_all[idx_tr10], ytr_all[idx_tr10]),
        "test": to_gpu(Xte, yte)}

def augment(x):
    B = x.shape[0]
    xp = F.pad(x, (4, 4, 4, 4), mode="reflect")
    i = torch.randint(0, 9, (B,), device=x.device); j = torch.randint(0, 9, (B,), device=x.device)
    ar = torch.arange(32, device=x.device)
    rows = (i[:, None] + ar)[:, None, :, None]; cols = (j[:, None] + ar)[:, None, None, :]
    x = xp[torch.arange(B, device=x.device)[:, None, None, None], torch.arange(3, device=x.device)[None, :, None, None], rows, cols]
    flip = torch.rand(B, device=x.device) < 0.5
    x[flip] = x[flip].flip(3)
    return x

class Preproc:
    def __init__(self, size, mean, std):
        self.size = size
        self.mean = torch.tensor(mean, device=DEV, dtype=torch.float32).view(1, 3, 1, 1)
        self.std = torch.tensor(std, device=DEV, dtype=torch.float32).view(1, 3, 1, 1)
    def __call__(self, xb_uint8, train_aug=False):
        x = xb_uint8.float() / 255.
        if train_aug: x = augment(x)
        if self.size != 32: x = F.interpolate(x, size=self.size, mode="bilinear", align_corners=False)
        return ((x - self.mean) / self.std).contiguous(memory_format=torch.channels_last)

PRE_CNN = Preproc(32, cifar_mean, cifar_std)
PRE_VGG = Preproc(VGG_RES, IMNET_MEAN, IMNET_STD)

## 3. Investigación: transfer learning en PyTorch

| Herramienta | Propósito | Parámetros relevantes |
|---|---|---|
| `torchvision.models.vgg16(weights=...)` | Construye VGG-16 (13 conv + 3 FC, ~138M params) y opcionalmente carga pesos. | `weights` (`None` = aleatorio), `progress`. |
| `VGG16_Weights` | Enum de pesos preentrenados (`IMAGENET1K_V1`, 71.6% top-1) con metadatos (categorías, métricas). | `.transforms()` devuelve el preprocesamiento exacto usado al entrenar: resize 256, center crop 224, `ToTensor`, normalize ImageNet. |
| `model.features` | `nn.Sequential` con los 5 bloques conv (conv3×3+ReLU, cierre con MaxPool). Índices: B1 `[0:5]`, B2 `[5:10]`, B3 `[10:17]`, B4 `[17:24]`, B5 `[24:31]`. | Salida 512×(H/32)×(W/32). |
| `model.avgpool` | `nn.AdaptiveAvgPool2d((7,7))`: produce siempre 512×7×7 sin importar la resolución de entrada -> permite usar 112×112 sin cambiar el clasificador. | `output_size`. |
| `model.classifier` | MLP: `Linear(25088,4096)-ReLU-Dropout-Linear(4096,4096)-ReLU-Dropout-Linear(4096,1000)`. Reemplazamos `classifier[6]` por `Linear(4096,10)`. | `in_features`, `out_features`, `p` del dropout. |
| `param.requires_grad` | Si es `False`, autograd no calcula gradientes para ese tensor (congelado) y no guarda activaciones para un prefijo congelado. Congelar un bloque: `for p in model.features[24:].parameters(): p.requires_grad = flag`. | Por tensor. |
| `model.train()` / `model.eval()` | Cambian el comportamiento de Dropout y BatchNorm (estadísticas batch vs running). | `mode`. |
| `torch.no_grad()` | Context manager que desactiva el grafo de autograd: menos memoria y más rápido en validación/inferencia. | - |
| `optimizer param_groups` | Lista de dicts `{"params": ..., "lr": ...}` que permite LR distintos por parte del modelo (backbone vs clasificador). | `lr`, `weight_decay` por grupo. |
| `thop.profile` / `torchinfo.summary` | Cuentan MACs y parámetros con un forward de prueba (thop registra hooks por capa). | `inputs`, `input_size`. |
| `torch.cuda.max_memory_allocated()` | Memoria pico asignada por tensores desde el último `reset_peak_memory_stats()`. | `device`. |
| `torch.cuda.synchronize()` | Espera a que terminen los kernels asíncronos de la GPU; necesario antes de leer `time.perf_counter()` para medir tiempos reales. | `device`. |

MACs vs FLOPs: un MAC (multiply-accumulate) es una multiplicación + una suma; por convención FLOPs ≈ 2 × MACs. thop reporta MACs.

Parámetros totales vs entrenables: totales = todos los pesos del modelo (memoria/tamaño en disco, costo de inferencia); entrenables = los que tienen `requires_grad=True`, que son los que reciben gradiente, ocupan estado del optimizador y determinan el costo del backward.

Costo de entrenamiento: el backward cuesta ≈ 2× el forward de las capas que lo ejecutan (gradiente respecto a la entrada y a los pesos). Con el backbone congelado no hay backward por él.

In [ ]:
_w = VGG16_Weights.IMAGENET1K_V1
print(_w.transforms())
print(vgg16(weights=None))

## 4. Modelos y utilidades de entrenamiento / medición

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, widths=(32, 64, 128), bn=True, dropout=0.0):
        super().__init__()
        layers, cin = [], 3
        for w in widths:
            for k in range(2):
                layers += [nn.Conv2d(cin if k == 0 else w, w, 3, padding=1, bias=not bn)]
                if bn: layers += [nn.BatchNorm2d(w)]
                layers += [nn.ReLU(inplace=True)]
            layers += [nn.MaxPool2d(2)]
            if dropout: layers += [nn.Dropout(dropout * 0.5)]
            cin = w
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(cin, 10))
    def forward(self, x):
        return self.head(self.features(x))

VGG_BLOCK_START = {1: 0, 2: 5, 3: 10, 4: 17, 5: 24}

def build_vgg(unfreeze_blocks=(), train_full_classifier=True, cls_dropout=0.5):
    m = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
    m.classifier[6] = nn.Linear(4096, 10)
    m.classifier[2].p = m.classifier[5].p = cls_dropout
    for p in m.features.parameters(): p.requires_grad = False
    for b in unfreeze_blocks:
        end = VGG_BLOCK_START.get(b + 1, 31)
        for p in m.features[VGG_BLOCK_START[b]:end].parameters(): p.requires_grad = True
    if not train_full_classifier:
        for p in m.classifier[:6].parameters(): p.requires_grad = False
    return m

def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)

def macs_of(module, size):
    m = copy.deepcopy(module).float().eval().to(DEV)
    macs, _ = profile(m, inputs=(torch.randn(1, 3, size, size, device=DEV),), verbose=False)
    return macs

def compute_cost(model, kind, size):
    total = macs_of(model, size)
    if kind == "cnn":
        bwd = total
    else:
        trainable_idx = [i for i, l in enumerate(model.features) if any(p.requires_grad for p in l.parameters())]
        start = trainable_idx[0] if trainable_idx else len(model.features)
        frozen = macs_of(model.features[:start], size) if start > 0 else 0
        bwd = total - frozen
    return total, bwd

@torch.no_grad()
def measure_latency(model, pre, n=100):
    model.eval(); x = pre(DATA["test"][0][:1])
    with torch.autocast("cuda", dtype=AMP_DTYPE):
        for _ in range(10): model(x)
        torch.cuda.synchronize(); t = time.perf_counter()
        for _ in range(n): model(x)
        torch.cuda.synchronize()
    return (time.perf_counter() - t) / n * 1000  # ms

@torch.no_grad()
def evaluate(model, split, pre, bs=250):
    model.eval(); X, y = DATA[split]; losses, preds = [], []
    for s in range(0, len(y), bs):
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            out = model(pre(X[s:s + bs]))
        losses.append(F.cross_entropy(out.float(), y[s:s + bs], reduction="sum").item())
        preds.append(out.argmax(1))
    preds = torch.cat(preds).cpu().numpy(); yt = y.cpu().numpy()
    p, r, f1, _ = precision_recall_fscore_support(yt, preds, average="macro", zero_division=0)
    return {"loss": sum(losses) / len(yt), "acc": accuracy_score(yt, preds), "prec": p, "rec": r, "f1": f1}, preds

In [ ]:
RUNS = {}

def train_run(name, family, build, cfg, split="train", log=True):
    torch.manual_seed(SEED)
    model = build().to(DEV).to(memory_format=torch.channels_last)
    pre = PRE_CNN if family == "cnn" else PRE_VGG
    tot, trn = count_params(model)
    # lr distinto para backbone y clasificador
    bb = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("features")]
    hd = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("features")]
    groups = [{"params": hd, "lr": cfg["lr"]}]
    if bb: groups.append({"params": bb, "lr": cfg.get("lr_backbone", cfg["lr"])})
    opt = torch.optim.AdamW(groups, weight_decay=cfg.get("wd", 5e-4))
    X, y = DATA[split]; bs, E = cfg["bs"], cfg["epochs"]
    steps = E * ((len(y) + bs - 1) // bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in groups], total_steps=steps, pct_start=0.2) \
        if cfg.get("sched") == "onecycle" else torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    crit = nn.CrossEntropyLoss(label_smoothing=cfg.get("ls", 0.0))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)
    hist = {k: [] for k in ["train_loss", "val_loss", "val_acc", "val_prec", "val_rec", "val_f1", "epoch_time"]}
    best, best_state = -1, None
    torch.cuda.reset_peak_memory_stats()
    for ep in range(E):
        model.train()
        torch.cuda.synchronize(); t0 = time.perf_counter()
        perm = torch.randperm(len(y), device=DEV); run_loss = torch.zeros((), device=DEV)
        for s in range(0, len(y), bs):
            b = perm[s:s + bs]
            xb = pre(X[b], train_aug=cfg.get("aug", True))
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                loss = crit(model(xb).float(), y[b])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.detach() * len(b)
        torch.cuda.synchronize(); dt = time.perf_counter() - t0
        m, _ = evaluate(model, "val", pre)
        hist["train_loss"].append(run_loss.item() / len(y)); hist["val_loss"].append(m["loss"])
        for k in ["acc", "prec", "rec", "f1"]: hist["val_" + k].append(m[k])
        hist["epoch_time"].append(dt)
        if m["f1"] > best:
            best = m["f1"]; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if log: print(f"[{name}] ep {ep+1}/{E} train {hist['train_loss'][-1]:.3f} val {m['loss']:.3f} acc {m['acc']:.4f} f1 {m['f1']:.4f} ({dt:.1f}s)")
    peak = torch.cuda.max_memory_allocated() / 2**20
    model.load_state_dict(best_state)
    model.cpu(); del opt; torch.cuda.empty_cache()  # si no se acaba la VRAM
    bi = int(np.argmax(hist["val_f1"]))
    rec = {"name": name, "family": family, "split": split, "cfg": cfg, "params_total": tot, "params_trainable": trn,
           "frozen": cfg.get("frozen", "-"), "hist": hist, "best_epoch": bi + 1,
           "best_val": {k: hist["val_" + k][bi] for k in ["acc", "prec", "rec", "f1"]},
           "time_per_epoch": float(np.mean(hist["epoch_time"])), "time_total": float(np.sum(hist["epoch_time"])),
           "peak_mem_mb": peak}
    RUNS[name] = rec
    json.dump(RUNS, open("results/runs.json", "w"), indent=1, default=float)
    return model, rec

### 4.a CNN desde cero (3 iteraciones)
- CNN-1: 3 bloques (32-64-128) sin BN, sin dropout, sin augmentation -> línea base.
- CNN-2: mismos 3 bloques + BatchNorm + dropout + augmentation.
- CNN-3: 4 bloques más anchos (64-128-256-512) + BN + dropout + aug + OneCycleLR.

In [ ]:
CNN_CFGS = {
    "CNN-1": dict(widths=(32, 64, 128), bn=False, dropout=0.0, aug=False, lr=1e-3, bs=256, epochs=10, sched="cosine"),
    "CNN-2": dict(widths=(32, 64, 128), bn=True, dropout=0.3, aug=True, lr=2e-3, bs=256, epochs=12, sched="cosine"),
    "CNN-3": dict(widths=(64, 128, 256, 512), bn=True, dropout=0.3, aug=True, lr=3e-3, bs=256, epochs=12, sched="onecycle"),
}
MODELS = {}
for n, c in CNN_CFGS.items():
    MODELS[n], _ = train_run(n, "cnn", lambda c=c: SimpleCNN(c["widths"], c["bn"], c["dropout"]), c)

### 4.b VGG-16 como feature extractor (3 iteraciones) - `features` 100% congelado
- FE-1: solo la nueva `Linear(4096,10)` entrenable.
- FE-2: todo `classifier` entrenable (FC1-FC3), LR 1e-3.
- FE-3: todo `classifier`, LR 3e-4 + label smoothing 0.1.

In [ ]:
FE_CFGS = {
    "FE-1": dict(full_cls=False, lr=1e-3, bs=128, epochs=3, frozen="features + FC1-FC2"),
    "FE-2": dict(full_cls=True, lr=1e-3, bs=128, epochs=3, frozen="features (B1-B5)"),
    "FE-3": dict(full_cls=True, lr=3e-4, ls=0.1, bs=128, epochs=3, frozen="features (B1-B5)"),
}
for n, c in FE_CFGS.items():
    MODELS[n], _ = train_run(n, "fe", lambda c=c: build_vgg((), c["full_cls"]), c)

### 4.c VGG-16 con fine-tuning (3 iteraciones) - LR del backbone menor que el del clasificador
- FT-1: bloque 5 descongelado, LR backbone 1e-4 / clasificador 1e-3.
- FT-2: bloques 4 y 5 descongelados, mismos LR (varía el nº de bloques).
- FT-3: bloques 4 y 5, LR backbone 1e-5 (efecto del LR del backbone).

In [ ]:
FT_CFGS = {
    "FT-1": dict(blocks=(5,), lr=1e-3, lr_backbone=1e-4, bs=128, epochs=3, frozen="B1-B4"),
    "FT-2": dict(blocks=(4, 5), lr=1e-3, lr_backbone=1e-4, bs=128, epochs=3, frozen="B1-B3"),
    "FT-3": dict(blocks=(4, 5), lr=1e-3, lr_backbone=1e-5, bs=128, epochs=3, frozen="B1-B3"),
}
for n, c in FT_CFGS.items():
    MODELS[n], _ = train_run(n, "ft", lambda c=c: build_vgg(c["blocks"], True), c)

### Tabla de iteraciones y curvas de pérdida

In [ ]:
def runs_table(names):
    rows = []
    for n in names:
        r = RUNS[n]
        rows.append({"iter": n, "congelado": r["frozen"],
                     "hparams": {k: v for k, v in r["cfg"].items() if k not in ("frozen",)},
                     "params tot (M)": r["params_total"] / 1e6, "entrenables (M)": r["params_trainable"] / 1e6,
                     "best ep": r["best_epoch"], **{"val " + k: v for k, v in r["best_val"].items()},
                     "s/epoch": r["time_per_epoch"], "t total (s)": r["time_total"], "mem pico (MB)": r["peak_mem_mb"]})
    return pd.DataFrame(rows)

ITERS = list(CNN_CFGS) + list(FE_CFGS) + list(FT_CFGS)
df_iter = runs_table(ITERS)
df_iter.to_csv("results/iteraciones.csv", index=False)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 250)
print(df_iter.drop(columns=["hparams"]).round(4).to_string())

fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, n in zip(axes.flat, ITERS):
    h = RUNS[n]["hist"]; ep = np.arange(1, len(h["train_loss"]) + 1)
    ax.plot(ep, h["train_loss"], "o-", label="train"); ax.plot(ep, h["val_loss"], "s-", label="val")
    ax.set_title(f"{n}  (val F1={RUNS[n]['best_val']['f1']:.3f})"); ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()
plt.tight_layout(); plt.savefig("results/figs/curvas_loss.png", dpi=120); plt.show()

## Evaluación final en test (una sola vez, mejor configuración de cada modelo)

In [ ]:
BEST = {fam: max([n for n in ITERS if RUNS[n]["family"] == fam], key=lambda n: RUNS[n]["best_val"]["f1"])
        for fam in ["cnn", "fe", "ft"]}
LABEL = {"cnn": "CNN desde cero", "fe": "VGG-16 FE", "ft": "VGG-16 FT"}
print("Mejores configuraciones:", BEST)
TEST, PREDS = {}, {}
for fam, n in BEST.items():
    TEST[fam], PREDS[fam] = evaluate(MODELS[n].to(DEV), "test", PRE_CNN if fam == "cnn" else PRE_VGG)
    MODELS[n].cpu(); torch.cuda.empty_cache()
    print(LABEL[fam], n, {k: round(v, 4) for k, v in TEST[fam].items()})

fig, axes = plt.subplots(1, 3, figsize=(19, 6))
yt = DATA["test"][1].cpu().numpy(); CM = {}
for ax, fam in zip(axes, BEST):
    CM[fam] = confusion_matrix(yt, PREDS[fam])
    ConfusionMatrixDisplay(CM[fam], display_labels=CLASSES).plot(ax=ax, colorbar=False, xticks_rotation=60)
    ax.set_title(f"{LABEL[fam]} ({BEST[fam]}) acc {TEST[fam]['acc']:.3f}")
plt.tight_layout(); plt.savefig("results/figs/confusion.png", dpi=110); plt.show()

def top_confusions(cm, k=4):
    c = cm.copy(); np.fill_diagonal(c, 0); s = c + c.T
    pairs = sorted(((s[i, j], CLASSES[i], CLASSES[j]) for i in range(10) for j in range(i + 1, 10)), reverse=True)[:k]
    return [f"{a}/{b}: {v}" for v, a, b in pairs]
CONF = {fam: top_confusions(CM[fam]) for fam in BEST}
for fam in BEST: print(LABEL[fam], CONF[fam])

## 4.1 Efecto de la cantidad de datos (10% del train, balanceado)
Re-entrenamos la mejor configuración de cada modelo con 4,500 imágenes. Como hay 10× menos pasos por epoch, aumentamos las epochs (CNN 25, VGG 6) para comparar con un presupuesto de optimización razonable; se selecciona la mejor epoch por F1 de validación y se evalúa en el mismo test.

In [ ]:
ALL_CFGS = {**CNN_CFGS, **FE_CFGS, **FT_CFGS}
def builder(n):
    c = ALL_CFGS[n]
    if n.startswith("CNN"): return lambda: SimpleCNN(c["widths"], c["bn"], c["dropout"])
    if n.startswith("FE"): return lambda: build_vgg((), c["full_cls"])
    return lambda: build_vgg(c["blocks"], True)

TEST10 = {}
for fam, n in BEST.items():
    c = dict(ALL_CFGS[n]); c["epochs"] = 25 if fam == "cnn" else 6
    m10, _ = train_run(n + "@10%", fam, builder(n), c, split="train10", log=False)
    TEST10[fam], _ = evaluate(m10.to(DEV), "test", PRE_CNN if fam == "cnn" else PRE_VGG)
    del m10; torch.cuda.empty_cache()
df_data = pd.DataFrame({LABEL[f]: {"acc 100%": TEST[f]["acc"], "acc 10%": TEST10[f]["acc"],
                                   "F1 100%": TEST[f]["f1"], "F1 10%": TEST10[f]["f1"],
                                   "caída F1 (pp)": 100 * (TEST[f]["f1"] - TEST10[f]["f1"])} for f in BEST}).T
df_data.to_csv("results/cantidad_datos.csv"); print(df_data.round(4))

## 5. Comparación de rendimiento y recursos

In [ ]:
COMP = {}
for fam, n in BEST.items():
    m, r = MODELS[n].to(DEV), RUNS[n]
    size = 32 if fam == "cnn" else VGG_RES
    macs, macs_bwd = compute_cost(m, "cnn" if fam == "cnn" else "vgg", size)
    lat = measure_latency(m, PRE_CNN if fam == "cnn" else PRE_VGG)
    m.cpu(); torch.cuda.empty_cache()
    n_img, epochs = 45000, r["cfg"]["epochs"]
    # backward ~ 2x forward de la parte entrenable
    train_flops = n_img * epochs * 2 * (macs + 2 * macs_bwd)
    COMP[fam] = {"config": n, "params tot (M)": r["params_total"] / 1e6, "params entren. (M)": r["params_trainable"] / 1e6,
                 "GMACs/img": macs / 1e9, "GFLOPs/img": 2 * macs / 1e9, "latencia (ms/img)": lat,
                 "s/epoch": r["time_per_epoch"], "epochs a mejor val": r["best_epoch"], "epochs totales": epochs,
                 "t total (s)": r["time_total"], "mem pico (MB)": r["peak_mem_mb"], "PFLOPs entren.": train_flops / 1e15,
                 **{"test " + k: TEST[fam][k] for k in ["acc", "prec", "rec", "f1"]},
                 "test acc 10%": TEST10[fam]["acc"], "test F1 10%": TEST10[fam]["f1"]}
df_comp = pd.DataFrame(COMP).T.rename(index=LABEL)
df_comp.to_csv("results/comparacion.csv"); print(df_comp.T.to_string())
json.dump({"hw": HW, "best": BEST, "test": TEST, "test10": TEST10, "conf": CONF,
           "comp": {k: {kk: (vv if isinstance(vv, str) else float(vv)) for kk, vv in v.items()} for k, v in COMP.items()}},
          open("results/summary.json", "w"), indent=1, default=float)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for fam, n in BEST.items():
    h = RUNS[n]["hist"]
    axes[0].plot(np.cumsum(h["epoch_time"]), h["val_acc"], "o-", label=f"{LABEL[fam]} ({n})")
axes[0].set_xlabel("tiempo de entrenamiento acumulado (s)"); axes[0].set_ylabel("accuracy validación"); axes[0].legend(); axes[0].grid(alpha=.3)
for fam in BEST:
    x = COMP[fam]["GFLOPs/img"]; y = COMP[fam]["test f1"]
    axes[1].scatter(x, y, s=80); axes[1].annotate(f"{LABEL[fam]}\n{COMP[fam]['PFLOPs entren.']:.0f} PFLOPs entren.", (x, y), textcoords="offset points", xytext=(6, -14), fontsize=8)
axes[1].set_xscale("log"); axes[1].set_xlabel("GFLOPs de inferencia por imagen (log)"); axes[1].set_ylabel("F1 macro test"); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("results/figs/rendimiento_vs_costo.png", dpi=120); plt.show()

In [ ]:
print(df_comp.T)